In [1]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch
from datetime import datetime
import io

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
# Retrieve account & region's info
session = boto3.Session()
account_id = session.client('sts').get_caller_identity().get('Account')
region = session.region_name

# SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = "forest-carbon-dung-processed-extended"

print(f"Account ID: {account_id}")
print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")

Account ID: 755352605209
Region: ca-central-1
Role: arn:aws:iam::755352605209:role/sagemaker-training-role
Bucket: forest-carbon-dung-processed-extended


In [3]:
# Count the total number of chips processed
s3 = boto3.client('s3')

def count_chips():
    total = 0
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix='processed/'):
        if 'Contents' in page:
            for obj in page['Contents']:
                if 'images/chip_' in obj['Key']:
                    total += 1
    return total

total_chips = count_chips()
print(f"Total chips available: {total_chips:,}")

Total chips available: 650,356


In [3]:
# Tạo một file trống để đánh dấu
dummy_data = "This is a dummy file to satisfy SageMaker's input requirement."

s3 = boto3.client('s3')
dummy_bucket = bucket  # Dùng chính bucket hiện tại
dummy_key = 'dummy/dummy_data.txt'

s3.put_object(Bucket=dummy_bucket, Key=dummy_key, Body=dummy_data)
print(f"Dummy file created at: s3://{dummy_bucket}/{dummy_key}")

Dummy file created at: s3://forest-carbon-dung-processed-extended/dummy/dummy_data.txt


In [15]:
estimator = PyTorch(
    entry_point="train.py",
    source_dir = './training_scripts',
    requirements_file = 'requirements.txt',
    role=role,
    instance_count=1,
    instance_type = 'ml.g5.xlarge',
    framework_version='2.0.1',
    py_version = "py310",
    output_path=f's3://{bucket}/models/',
    hyperparameters={
        'epochs': 50,
        'batch_size': 32,
        'learning_rate': 0.001,
        'model_type': 'unet',
        's3-bucket': bucket
    },
    max_run=172800,
    base_job_name=f'forest-segmentation-{datetime.now().strftime("%Y%m%d-%H%M")}'
)
print(f"✅ Estimator created successfully!")
print(f"   Job name prefix: {estimator._current_job_name}")
print(f"   Instance type: ml.g5.xlarge")


✅ Estimator created successfully!
   Job name prefix: None
   Instance type: ml.g5.xlarge


In [ ]:
import io
print(f"🚀 Starting training job...")
print(f"⌛ This will take 15-20 hours long")

# Chạy training
estimator.fit(
    inputs={
        'train': f's3://{bucket}/dummy/'
    },
    wait=True,   # Đợi job xong
    logs=True    # In log real-time
)

print(f"\n✅ Training completed!")
print(f"📁 Model saved at: {estimator.output_path}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


🚀 Starting training job...
⌛ This will take 15-20 hours long


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: forest-segmentation-20260427-2323-2026-04-27-23-24-02-797


2026-04-27 23:24:08 Starting - Starting the training job
2026-04-27 23:24:08 Pending - Training job waiting for capacity...
2026-04-27 23:24:22 Pending - Preparing the instances for training...
2026-04-27 23:24:47 Downloading - Downloading input data.

In [8]:
import boto3

logs = boto3.client('logs')

LOG_GROUP = '/aws/sagemaker/TrainingJobs'

# Liệt kê tất cả log stream có tiền tố 'forest-segmentation-20260427-2150'
try:
    response = logs.describe_log_streams(
        logGroupName=LOG_GROUP,
        logStreamNamePrefix='forest-segmentation-20260427-2150',
        descending=True,
        limit=10
    )
    
    print("Các log stream tìm thấy:")
    for stream in response['logStreams']:
        print(f"  - {stream['logStreamName']}")
        print(f"    Last event time: {stream.get('lastEventTimestamp', 'N/A')}")
        
    # Nếu tìm thấy, tự động lấy cái đầu tiên
    if response['logStreams']:
        log_stream = response['logStreams'][0]['logStreamName']
        print(f"\nSẽ lấy log từ stream: {log_stream}")
        
        events = logs.get_log_events(
            logGroupName=LOG_GROUP,
            logStreamName=log_stream,
            limit=50,
            startFromHead=False
        )
        
        print("\n=== 50 dòng log cuối cùng ===\n")
        for event in reversed(events['events']):
            print(event['message'])
    else:
        print("Không tìm thấy log stream nào")

except Exception as e:
    print(f"Lỗi: {e}")

Các log stream tìm thấy:
  - forest-segmentation-20260427-2150-2026-04-27-21-50-27-954/algo-1-1777326720
    Last event time: 1777326967910

Sẽ lấy log từ stream: forest-segmentation-20260427-2150-2026-04-27-21-50-27-954/algo-1-1777326720

=== 50 dòng log cuối cùng ===

